# Exploring CTA bus ridership

This notebook explores CTA bus ridership since 2001, understanding the available data and the system as a whole. 

After exploration, it also generates datasets used in other notebooks. (Run this notebook (it writes `data/derived/`), before `holidays.ipynb` and `seasonality.ipynb`).

Exploration is a first step before setting up a before/ after comparison for the 10 minute frequent network program,
an analysis which which will be fleshed out in `frequent_network_analysis.ipynb` (currently under construction/ being rebuilt).


### Outline
0. Import packages and Load in the data
    - CTA daily bus ridership dataset: [CTA Ridership – Bus Routes – Daily Totals by Route](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Daily-Totals-by-Route/jyb9-n7fm/about_data)
    - CTA monthly bus ridership dataset:   [CTA – Ridership – Bus Routes – Monthly Day-Type Averages & Totals](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Monthly-Day-Type-Averages/bynn-gwxy)
1. Data quality checks and cross checks
   - missing data, NANs, etc.
   - cross check datasets
   - define/check weeks and look for endpoint issues
2. Define cooridors: how do we treat routes that cover the same streets?
    - take the same base number, look at the family
    - ignore R routes from 2013 redline project
3. Explore total ridership for the full dataset, 2001-2026
4. Explore ridership by route:
   - which routes have highest ridership
   - route change over time: routes that have started/ended; route growth or shrinkage relative to system average
   - route recovery since pandemic
5. Explore ridership by day of the week
6. Save the cleaned data for the companion notebooks

Two topics have their own notebooks:

- **`holidays.ipynb`** — which days CTA actually runs a holiday schedule, and what that does to ridership.
- **`seasonality.ipynb`** — the within-year profile, with the trend removed and holiday weeks held out.


### Notes about the data
 Coming soon.

### Open questions/ future improvements
- think more carefully about how express / branch routes (`X49`, `53A`, …) fold into corridors.
-
- Before/after window lengths for the event study.
- The definition of "usual" used in `holidays.ipynb` — flagged inline there.

## 0. Setup and load the data

### 0.a) Import necessary packages and define general plotting parameters

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D

# Okabe-Ito: the published colour-vision-deficiency-safe categorical set.
BLUE, ORANGE, GREEN, PURPLE = '#0072B2', '#D55E00', '#009E73', '#CC79A7'
GRAY, INK = '#C9C9C9', '#333333'

# Ridership regimes. 2020 gets its own colour because it is not comparable to anything
# else; the two recovery eras are lighter shades of it because the system has not
# returned to the pre-2020 level.
ERAS = [('pre-2020',     2001, 2019, BLUE),
        ('2020',         2020, 2020, ORANGE),
        ('2021-2022',    2021, 2022, '#EE8A4E'),
        ('2023-present', 2023, 2026, '#F5BE99')]
ERA_ORDER = [e[0] for e in ERAS]
ERA_COLOR = {e[0]: e[3] for e in ERAS}

def era(year):
    """Map a calendar year to its ridership regime."""
    for name, lo, hi, _ in ERAS:
        if lo <= year <= hi:
            return name
    return None

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#9A9A9A', 'axes.grid': True,
    'grid.color': '#E8E8E8', 'grid.linewidth': 0.8,
})
fmt_riders = FuncFormatter(lambda v, _: f'{v*1e-6:.1f}M' if v >= 1e6 else f'{v*1e-3:.0f}k')

### 0.b) Read in the main CTA bus data file

[CTA Ridership – Bus Routes – Daily Totals by Route](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Daily-Totals-by-Route/jyb9-n7fm/about_data)
  (Chicago Data Portal, dataset `jyb9-n7fm`), pulled via the Socrata API:
  `https://data.cityofchicago.org/resource/jyb9-n7fm.csv`.
  Fields: `route`, `date`, `daytype` (**W** = weekday, **A** = Saturday, **U** =
  Sunday/holiday), `rides`.

```
Re-download if needed:
!curl -s "https://data.cityofchicago.org/resource/jyb9-n7fm.csv?\$limit=2000000&\$order=route,date" -o data/cta_bus_daily.csv
```

In [ ]:
RAW = pd.read_csv('data/cta_bus_daily.csv', dtype={'route': str, 'daytype': str})
print(f'rows read : {len(RAW):,}')
print(f'columns   : {list(RAW.columns)}')
RAW.head()

### 0.c) Read in *Monthly Averages and Totals File* to check against and get route names

The daily file doesn't have route names, but here is another file that aggregates by month on the same portal, along
with day-type averages we can check our own aggregation against:

[CTA – Ridership – Bus Routes – Monthly Day-Type Averages & Totals](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Monthly-Day-Type-Averages/bynn-gwxy)
(`bynn-gwxy`), saved to `data/cta_bus_monthly.csv`:

```
curl "https://data.cityofchicago.org/resource/bynn-gwxy.csv?$limit=50000&$order=route,month_beginning" \
     -o data/cta_bus_monthly.csv
```

Columns: `route`, `routename`, `month_beginning`, `avg_weekday_rides`, `avg_saturday_rides`,
`avg_sunday_holiday_rides`, `monthtotal`.

In [ ]:
MON = pd.read_csv('data/cta_bus_monthly.csv', dtype={'route': str},
                  parse_dates=['month_beginning'])
print(f'rows {len(MON):,}   routes {MON.route.nunique()}   '
      f'{MON.month_beginning.min().date()} .. {MON.month_beginning.max().date()}')

# Names change over time, so take each route's most recent name and list the changes.
latest = MON.sort_values('month_beginning').groupby('route').routename.last()
changed = MON.groupby('route').routename.nunique()
changed = changed[changed > 1]
print(f'\nroutes renamed at least once: {len(changed)}')
for r in changed.index:
    print(f'  {r:<5} ' + ' -> '.join(MON.loc[MON.route == r, 'routename'].unique()))

## 1. Data quality checks and cross checks

Nothing is dropped or filtered in this section — it only counts. Every check prints its
result even when the result is zero, so an absent problem is visible rather than assumed.

### 1.a) Missing data, NaNs, duplicates

In [ ]:
# ---------------------------------------------------------------------------
# INTEGRITY REPORT.  What weird stuff is in the data?
# ---------------------------------------------------------------------------
d = RAW.copy()
d['date']  = pd.to_datetime(d.date,  errors='coerce')
d['rides'] = pd.to_numeric(d.rides, errors='coerce')

for label, n in {
    'rows':                len(d),
    'unparseable dates':   int(d.date.isna().sum()),
    'non-numeric rides':   int(d.rides.isna().sum()),
    'rides < 0':           int((d.rides < 0).sum()),
    'rides == 0':          int((d.rides == 0).sum()),
    'null route':          int(d.route.isna().sum()),
    'null daytype':        int(d.daytype.isna().sum()),
}.items():
    print(f'{label:<28}: {n:>10,}')

print()
print(f'{"date range":<28}: {d.date.min().date()} .. {d.date.max().date()}')
print(f'{"distinct routes":<28}: {d.route.nunique():>10,}')
print(f'{"daytype values":<28}: {sorted(d.daytype.dropna().unique())}')

dup = d.duplicated(subset=['route', 'date'], keep=False)
print(f'{"duplicate (route,date) rows":<28}: {int(dup.sum()):>10,}')
if dup.any():
    g = d.loc[dup].groupby(['route', 'date'])
    print(f'{"  affected route-days":<28}: {g.ngroups:>10,}')
    print(f'{"  identical rides in dup":<28}: {int((g.rides.nunique() == 1).sum()):>10,}')
    display(d.loc[dup].sort_values(['route', 'date']).head(20))
    print(f'  (showing up to 20 of {int(dup.sum()):,} duplicate rows)')

In [ ]:
# Calendar coverage: is every day between the first and last date present?
days    = pd.date_range(d.date.min(), d.date.max(), freq='D')
missing = days.difference(pd.Index(d.date.unique()))
print(f'{"calendar days in range":<28}: {len(days):>10,}')
print(f'{"days with no rows at all":<28}: {len(missing):>10,}')
if len(missing):
    print('  ', [str(x.date()) for x in missing[:20]],
          f'... (showing up to 20 of {len(missing)})')

per_day = d.groupby('date').route.nunique()
print(f'\nroutes reporting per day:  min={per_day.min()}   '
      f'median={per_day.median():.0f}   max={per_day.max()}')

### 1.b) Cross-check the daily file against the monthly file

`daytype` `W` excludes holidays, so a month's mean over `W` days should equal the published
`avg_weekday_rides`. Any systematic gap would mean we are reading the day types wrongly.

In [ ]:
ours = (d[d.daytype == 'W']
          .groupby(['route', pd.Grouper(key='date', freq='MS')]).rides.mean()
          .rename('ours').reset_index()
          .rename(columns={'date': 'month_beginning'}))

chk = ours.merge(MON[['route', 'month_beginning', 'avg_weekday_rides']],
                 on=['route', 'month_beginning'], how='inner')
chk['diff_pct'] = (chk.ours - chk.avg_weekday_rides) / chk.avg_weekday_rides * 100

print(f'month-route pairs compared : {len(chk):,}')
print(f'  exact to within 0.5%     : {int((chk.diff_pct.abs() < 0.5).sum()):,}')
print(f'  median |difference|      : {chk.diff_pct.abs().median():.3f}%')
print(f'  worst |difference|       : {chk.diff_pct.abs().max():.2f}%')
print('\nlargest disagreements:')
print(chk.reindex(chk.diff_pct.abs().sort_values(ascending=False).index)
         .head(8)[['route', 'month_beginning', 'ours', 'avg_weekday_rides', 'diff_pct']]
         .to_string(index=False))

# Which routes appear in one file but not the other?
print(f'\nin daily but not monthly (no name available): '
      f'{sorted(set(d.route) - set(MON.route))}')
print(f'in monthly but not daily: {sorted(set(MON.route) - set(d.route))}')

### 1.c) Weeks, and endpoint issues

Weeks are **Mon–Sun**, labelled by the Monday that starts them, derived from `date` alone.
Each week carries `days` = how many distinct calendar days actually appear in the data, so a
short week at either end of the record is visible rather than silently reading as a dip.

In [ ]:
d['week'] = d.date - pd.to_timedelta(d.date.dt.weekday, unit='D')
d['dow']  = d.date.dt.dayofweek                 # 0=Mon .. 6=Sun, from the date itself
DOW = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

wk = (d.groupby('week')
        .agg(rides=('rides', 'sum'), days=('date', 'nunique'), routes=('route', 'nunique'))
        .reset_index())
wk['partial'] = wk.days < 7

print(f'weeks spanned            : {len(wk):,}')
print(f'partial weeks (<7 days)  : {int(wk.partial.sum())}')
print()
print(wk[wk.partial].to_string(index=False))

## 2. Corridors: how do we treat routes that cover the same streets?

**Corridor**, starting definition: routes sharing the same numeric root — so `49`, `X49`, `49B`
belong to corridor `49`, and `J14` to corridor `14`. This is a first cut, not a final answer;
branch and express routes do not always follow the same street, so the families printed below
are meant to be audited one at a time before any of them are actually summed.

In [ ]:
import re

def corridor(route):
    """Numeric root of a route id: X49 -> 49, J14 -> 14, 1001 -> 1001."""
    m = re.search(r'\d+', route)
    return m.group() if m else route

d['name']     = d.route.map(latest)
d['corridor'] = d.route.map(corridor)

size = d.groupby('route').rides.mean()
fam  = (d.groupby('corridor').route.unique()
          .loc[lambda s: s.map(len) > 1]
          .sort_index(key=lambda i: i.astype(int)))

print(f'corridors: {d.corridor.nunique()}   of which multi-route: {len(fam)}')
print('Read this list critically -- some of these are one street, others are not.\n')
for cor, routes in fam.items():
    print(f'  corridor {cor}')
    for r in sorted(routes, key=lambda x: -size[x]):
        print(f'      {r:<6} {size[r]:>8,.0f}/day   {latest.get(r, "(no name)")}')

## 3. Total ridership for the full dataset, 2001-2026

By week. The lower panel is the number of routes reporting that week. The system total is a sum
over a route set that changes over time, so the two have to be read together.

In [ ]:
wk['era'] = wk.week.dt.year.map(era)

fig, (ax, ax2) = plt.subplots(2, 1, figsize=(11, 5.8), sharey=False, sharex=True,
                              gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.12})

# Eras are contiguous in time, so extend each slice by one week to close the seams.
for name in ERA_ORDER:
    idx = np.flatnonzero((wk.era == name).to_numpy())
    sl = slice(idx[0], idx[-1] + 2)
    ax.plot(wk.week[sl], wk.rides[sl], lw=1.0, color=ERA_COLOR[name], label=name)

# Only draw the partial-week marker if there are any; the count is printed above either way.
# Drawn before the legend is built so that its label actually appears in the legend.
if wk.partial.any():
    ax.scatter(wk.loc[wk.partial, 'week'], wk.loc[wk.partial, 'rides'],
               s=30, color=PURPLE, zorder=3, label='partial week (<7 days of data)')

ax.set_ylabel('rides per week')
ax.yaxis.set_major_formatter(fmt_riders)
ax.set_title('CTA bus ridership by week — entire dataset', loc='left', fontsize=11)
ax.legend(frameon=False, loc='lower left', ncol=4)

ax2.plot(wk.week, wk.routes, lw=1.0, color=INK)
ax2.set_ylabel('routes\nreporting')
ax2.set_xlabel('week (Monday)')
plt.show()

## 4. Explore ridership by route

### 4.a) Which routes have the highest ridership

Every route in the dataset by week, one line each. The 20 Frequent Network routes are highlighted.

Log scale, because route sizes span orders of magnitude. Nothing is excluded.

> **Note.** Only the 20 route IDs as CTA labels them are highlighted. Their express / branch
> variants (`X49`, `53A`, …) are drawn in grey with everything else, because the corridor
> question is still open.

In [ ]:
FREQ = ['J14', '4', '9', '12', '20', '34', '47', '49', '53', '54',
        '55', '60', '63', '66', '72', '77', '79', '81', '82', '95']

in_data = set(d.route.unique())
present = [r for r in FREQ if r in in_data]
print(f'Frequent Network routes (CTA labelling) : {len(FREQ)}')
print(f'  found in the data                     : {len(present)}')
print(f'  NOT found in the data                 : {[r for r in FREQ if r not in in_data]}')

rw = d.pivot_table(index='week', columns='route', values='rides', aggfunc='sum')
print(f'\nroute x week matrix: {rw.shape[0]:,} weeks x {rw.shape[1]:,} routes')
print(f'empty cells (route not reporting that week): {int(rw.isna().sum().sum()):,}'
      f'  of {rw.size:,}')

In [ ]:
others = [r for r in rw.columns if r not in present]

fig, ax = plt.subplots(figsize=(11, 5.6))
ax.plot(rw.index, rw[others],  lw=0.5, color=GRAY,   alpha=0.55)
ax.plot(rw.index, rw[present], lw=0.9, color=ORANGE, alpha=0.85)
ax.set_yscale('log')
ax.set_ylabel('rides per week (log scale)')
ax.set_xlabel('week (Monday)')
ax.set_title('Every bus route by week — Frequent Network routes highlighted',
             loc='left', fontsize=11)
ax.legend(handles=[Line2D([], [], color=ORANGE, lw=1.6, label=f'Frequent Network ({len(present)})'),
                   Line2D([], [], color=GRAY,   lw=1.6, label=f'all other routes ({len(others)})')],
          frameon=False, loc='lower left')
plt.show()

#### Route inventory and ridership statistics

Unit throughout is **riders per day** — the raw records are daily totals, so a route's mean is
its mean over the days it reported. Weekdays, Saturdays and Sundays are pooled here, so a
route's mean reflects its weekday/weekend mix as well as its size.

`status` marks routes that stopped reporting more than 30 days before the end of the data —
these are routes that were cut or renumbered. Nothing is filtered; all 188 routes are listed.

In [ ]:
END = d.date.max()
g = d.groupby('route')

inv = pd.DataFrame({
    'name':     g.name.first(),
    'corridor': g.corridor.first(),
    'first':    g.date.min(),
    'last':     g.date.max(),
    'days':     g.date.nunique(),
    'mean':     g.rides.mean(),
    'median':   g.rides.median(),
    'std':      g.rides.std(),
    'min':      g.rides.min(),
    'max':      g.rides.max(),
})
inv['min_date'] = d.loc[g.rides.idxmin(), ['route', 'date']].set_index('route').date
inv['max_date'] = d.loc[g.rides.idxmax(), ['route', 'date']].set_index('route').date
inv['status']   = np.where(inv['last'] >= END - pd.Timedelta(days=30),
                           'active', 'ended ' + inv['last'].dt.strftime('%Y-%m'))

inv = inv.sort_values('mean', ascending=False)
for col in ('first', 'last', 'min_date', 'max_date'):
    inv[col] = inv[col].dt.strftime('%Y-%m-%d')

print(f'routes: {len(inv)}   active: {int((inv.status == "active").sum())}   '
      f'ended: {int((inv.status != "active").sum())}')
with pd.option_context('display.max_rows', None, 'display.width', 200):
    display(inv[['name', 'corridor', 'first', 'last', 'days', 'status', 'mean', 'median',
                 'std', 'min', 'min_date', 'max', 'max_date']].round(1))

The same table as a plot: every route ranked by mean riders per day, largest first.

In [ ]:
FREQ_SET = set(FREQ)
is_freq = inv.index.isin(FREQ_SET)
rank = np.arange(1, len(inv) + 1)

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.scatter(rank[~is_freq], inv['mean'][~is_freq], s=14, color=GRAY,
           label=f'other routes ({int((~is_freq).sum())})')
ax.scatter(rank[is_freq], inv['mean'][is_freq], s=22, color=ORANGE,
           label=f'Frequent Network ({int(is_freq.sum())})', zorder=3)
for k, r in enumerate(inv.index[:8]):                       # label the eight largest
    ax.annotate(f'{r} {inv.loc[r, "name"]}', (rank[inv.index.get_loc(r)], inv.loc[r, 'mean']),
                xytext=(8, 9 if k % 2 else -9), textcoords='offset points',
                fontsize=7, va='center')
ax.set_xlabel('rank'); ax.set_ylabel('mean riders/day')
ax.yaxis.set_major_formatter(fmt_riders)
ax.set_title('Route size, ranked — whole dataset', loc='left', fontsize=11)
ax.legend(frameon=False, loc='upper right')
plt.show()

share = inv['mean'].sort_values(ascending=False).cumsum() / inv['mean'].sum()
print(f'top 20 routes carry {share.iloc[19]:.0%} of mean daily boardings; '
      f'top 50 carry {share.iloc[49]:.0%}')

### 4.b) Route change over time

Same statistics split by era. A route absent from an era simply did not run then — those cells
are blank rather than zero.

> **Deviation to flag:** you suggested `pre-2020 / 2020-2022 / 2023-present`. I split 2020 out
> on its own, because pooling it with 2021–2022 hides how different it was. Collapse the two
> middle columns if you'd rather have your original grouping.

In [ ]:
d['era'] = d.date.dt.year.map(era)

per_era = (d.groupby(['route', 'era']).rides
             .agg(['mean', 'median', 'std', 'size'])
             .rename(columns={'size': 'days'})
             .unstack('era')
             .reindex(columns=ERA_ORDER, level=1))

per_era = per_era.reindex(inv.index)            # keep the sort by overall mean
print(f'2023-present is a partial era: {d[d.era == "2023-present"].date.max().date()} is the '
      f'last date, so 2026 contributes Jan-May only.')
with pd.option_context('display.max_rows', None, 'display.width', 200):
    display(per_era.round(1))

In [ ]:
# When each route ran. Sorted by start then end date, so cuts show up as a diagonal.
span = (d.groupby('route').date.agg(['min', 'max'])
          .join(inv[['status', 'mean', 'name']])
          .sort_values(['min', 'max']))
ended = span.status != 'active'

fig, ax = plt.subplots(figsize=(11, 6.0))
y = np.arange(len(span))
ax.hlines(y[~ended], span['min'][~ended], span['max'][~ended], color=BLUE, lw=1.6,
          label=f'active ({int((~ended).sum())})')
ax.hlines(y[ended], span['min'][ended], span['max'][ended], color=ORANGE, lw=1.6,
          label=f'ended ({int(ended.sum())})')
ax.scatter(span['max'][ended], y[ended], s=9, color=ORANGE, zorder=3)

ax.set_yticks([]); ax.set_ylabel(f'route  (n={len(span)}, ordered by start date)')
ax.set_xlabel('date')
ax.set_title('Route lifespans — orange routes stopped reporting', loc='left', fontsize=11)
ax.legend(frameon=False, loc='lower right', bbox_to_anchor=(1, 1.0), ncol=2)
plt.show()

print(f'the {int(ended.sum())} routes that stopped reporting, largest first:')
print(span[ended].nlargest(20, 'mean')[['name', 'min', 'max', 'mean']]
        .assign(min=lambda t: t['min'].dt.strftime('%Y-%m'),
                max=lambda t: t['max'].dt.strftime('%Y-%m'))
        .rename(columns={'min': 'first', 'max': 'last', 'mean': 'riders/day'})
        .round(0).to_string())
print(f'(showing 20 of {int(ended.sum())})')

### 4.c) Route recovery since the pandemic

Each route's mean riders per day in 2025 — the last complete year — against its own pre-2020
mean. Both axes are log, so equal distances are equal ratios.

In [ ]:
# Recovery: pre-2020 mean against calendar 2025, the last complete year.
y2025 = d[d.date.dt.year == 2025].groupby('route').rides.mean()
rec = pd.DataFrame({'pre': per_era[('mean', 'pre-2020')],
                    'now': y2025,
                    'name': inv.name}).dropna(subset=['pre', 'now'])
rec['ratio'] = rec.now / rec.pre
freq_mask = rec.index.isin(FREQ_SET)

fig, ax = plt.subplots(figsize=(6.6, 6.2))
lims = [rec[['pre', 'now']].to_numpy().min() * 0.7, rec[['pre', 'now']].to_numpy().max() * 1.4]
ax.plot(lims, lims, color=INK, lw=0.9, ls=':', label='no change')
ax.plot(lims, [v * 0.5 for v in lims], color=GRAY, lw=0.9, ls='--', label='half of pre-2020')
ax.scatter(rec.pre[~freq_mask], rec.now[~freq_mask], s=16, color=GRAY)
ax.scatter(rec.pre[freq_mask], rec.now[freq_mask], s=26, color=ORANGE, zorder=3,
           label='Frequent Network')
for r in rec.ratio.nlargest(4).index.union(rec.ratio.nsmallest(4).index):
    ax.annotate(r, (rec.loc[r, 'pre'], rec.loc[r, 'now']), xytext=(5, 3),
                textcoords='offset points', fontsize=7)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('mean riders/day, pre-2020'); ax.set_ylabel('mean riders/day, 2025')
ax.set_title('Recovery by route — 2025 against pre-2020', loc='left', fontsize=11)
ax.legend(frameon=False, loc='upper left')
plt.show()

print(f'routes above pre-2020 level : {int((rec.ratio > 1).sum())} of {len(rec)}')
print(f'median recovery ratio       : {rec.ratio.median():.2f}')
print('\nstrongest and weakest:')
print(pd.concat([rec.nlargest(5, "ratio"), rec.nsmallest(5, "ratio")])
        [['name', 'pre', 'now', 'ratio']].round(2).to_string())

## 5. Explore ridership by day of the week

Three views, because "by day of week" means different things at different scales:

- **(a)** absolute mean rides per day, one line per year — dominated by the level change.
- **(b)** the same lines divided by each year's own mean, which is the test of whether the
  *shape* is stable year to year.
- **(c)** the spread within a single recent year, so the shape in (b) can be read against how
  much an individual day actually moves.

In [ ]:
day = d.groupby('date', as_index=False).rides.sum()
day['dow']  = day.date.dt.dayofweek
day['year'] = day.date.dt.year
day['era']  = day.year.map(era)

by_yr = day.pivot_table(index='year', columns='dow', values='rides', aggfunc='mean')
shape = by_yr.div(by_yr.mean(axis=1), axis=0)      # each year normalised by its own mean
YR = 2025                                          # last complete calendar year

fig, axes = plt.subplots(1, 3, figsize=(13, 4.0))

for yr in by_yr.index:
    col = ERA_COLOR[era(yr)]
    axes[0].plot(range(7), by_yr.loc[yr], color=col, lw=1.2)
    axes[1].plot(range(7), shape.loc[yr], color=col, lw=1.2)

axes[0].set_title('(a) mean rides per day', loc='left', fontsize=10)
axes[0].yaxis.set_major_formatter(fmt_riders)
axes[1].set_title("(b) shape: divided by each year's mean", loc='left', fontsize=10)
axes[1].axhline(1, color=INK, lw=0.8, ls=':')
axes[1].legend(handles=[Line2D([], [], color=ERA_COLOR[n], lw=1.6, label=n) for n in ERA_ORDER],
               frameon=False, fontsize=8, loc='lower left')

sub = day[day.year == YR]
for i in range(7):
    v = sub.loc[sub.dow == i, 'rides']
    axes[2].scatter(i + np.random.uniform(-.16, .16, len(v)), v,
                    s=7, color=BLUE, alpha=0.35, linewidths=0)
    lo, mid, hi = np.percentile(v, [10, 50, 90])
    axes[2].plot([i - .3, i + .3], [mid, mid], color=ORANGE, lw=2, zorder=3)
    axes[2].plot([i, i], [lo, hi], color=ORANGE, lw=1, zorder=3)
axes[2].set_title(f'(c) every day in {YR}: median, 10-90th pct', loc='left', fontsize=10)
axes[2].yaxis.set_major_formatter(fmt_riders)

for ax in axes:
    ax.set_xticks(range(7)); ax.set_xticklabels(DOW)
plt.show()

**On the statistic.** Max-minus-min across years is a poor summary — it is decided by whichever
single year is most extreme, which here is 2020. Standard deviation and inter-quartile range
across years are reported instead, and separately by era, so a genuinely stable shape can be
told apart from one held together by averaging.

In [ ]:
def spread(frame):
    """Across-year spread of the normalised day-of-week shape."""
    return pd.DataFrame({
        'std':     frame.std(),
        'IQR':     frame.quantile(.75) - frame.quantile(.25),
        'max-min': frame.max() - frame.min(),
    }).rename(index=lambda i: DOW[i]).T

print('all years (2001-2026)')
print(spread(shape).round(3).to_string())

for name in ERA_ORDER:
    yrs = [y for y in shape.index if era(y) == name]
    print(f'\n{name}  (n={len(yrs)} years)')
    print('  single year - no across-year spread defined' if len(yrs) < 2
          else spread(shape.loc[yrs]).round(3).to_string())

## 6. Save the cleaned data for the companion notebooks

`d` picks up its derived columns across sections 1–4, so it is written once, here, rather than
in pieces. `holidays.ipynb` and `seasonality.ipynb` read these files instead of re-deriving
them. `data/` is gitignored, so nothing here is committed.

In [ ]:
from pathlib import Path

DERIVED = Path('data/derived')
DERIVED.mkdir(parents=True, exist_ok=True)

daily_path = DERIVED / 'daily.csv'
d.to_csv(daily_path, index=False)
print(f'{str(daily_path):<32} {len(d):>10,} rows   '
      f'{daily_path.stat().st_size / 1e6:.0f} MB')
print(f'  columns: {list(d.columns)}')

inv_path = DERIVED / 'route_inventory.csv'
inv.to_csv(inv_path)
print(f'{str(inv_path):<32} {len(inv):>10,} rows')
print(f'  columns: {list(inv.columns)}')

## Where this leaves us

### Established

- The data is clean: no duplicates, no gaps, no missing days, and our day-type handling
  reproduces the published monthly averages to within 0.001% (§1.b).
- Day-of-week shape is stable to ~1-2% within an era (§5).

### Decisions still open

1. **The corridor rule.** Shared numeric root is a starting point, but the R-prefixed routes are
   2013 Red Line reconstruction shuttles and do not belong in the corridors they would join.
2. **How express / branch routes fold into corridors** more generally (`X49`, `53A`, …).

### Continued elsewhere

- **`holidays.ipynb`** — CTA's operational holiday list and the size of the holiday effect.
- **`seasonality.ipynb`** — the within-year profile, and whether it can be pooled across eras.